# `Matplotlib Plots For Newbies: using Sales Data`

# <font color=red>Mr Fugu Data Science</font>

# (????)

**`Help Support The Channel: Buy Me A Coffee @mrfugudatasci`**

In [ ]:
# dataframe operations
import pandas as pd
# general plotting
import matplotlib as plt
# import files and datetime formatting, json parsing
import os,datetime,json
# general plot formatting
import matplotlib.pyplot as plt
#format plot x/y tick labels
from matplotlib.ticker import FuncFormatter

%matplotlib inline

In [ ]:
# Find a file outside your directory:

def os_any_dir_search(file):
    u=[]
    for p,n,f in os.walk(os.getcwd()):
        
        for a in f:
            a = str(a)
            if a.endswith(file): # can be (.csv) or a file like I did and search 
#                 print(a)
#                 print(p)
                t=pd.read_csv(p+'/'+file)
                u.append(p+'/'+a)
    return t,u

In [ ]:
retail_sales_cleaned = os_any_dir_search('SalesDataNoCancels.csv')[0]

retail_sales_cleaned.head()

In [ ]:
# Sales Totals:
retail_sales_cleaned['Sales_Total']=\
retail_sales_cleaned['Quantity']*retail_sales_cleaned['UnitPrice']

# `Ex.1): Top 10 items sold with corresponding country`

we need to do groupby and sorting to find this criteria. But, we need to do this in 2 steps.

In [ ]:
subset_item_dta=retail_sales_cleaned.loc[:,['Country',
            'Description','Sales_Total']]
subset_item_dta.groupby(['Country','Sales_Total']).sum().\
sort_values('Sales_Total',ascending=False)[:10]
# subset_item_dta.groupby(['Country','Sales_Total']).sum()

In [ ]:
gg=subset_item_dta.groupby(['Country','Sales_Total']).sum().\
sort_values('Sales_Total',ascending=False)[:10]

In [ ]:
# this is a multindex object: 
list(gg.index)

In [ ]:
ten_items=[]
for i in range(len(gg['Description'])):
    ten_items.append(gg['Description'][i])
ten_items

In [ ]:
# Top 10 sales by country
cntry_Sales=pd.DataFrame(list(gg.index),columns=['Country','Sales'])
# subset of the 10 items we want
top_ten_items_sorted=pd.DataFrame(ten_items,columns=['Items'])

# Top 10 items, with their sales and what country came from:
ff=pd.concat([cntry_Sales,top_ten_items_sorted],axis=1)

ff.sort_values('Country')

In [ ]:
# Creating nested list with our sales by country
store_=[]

ff_sort=ff.sort_values('Country')

names_=ff_sort['Country'].unique()
for i in names_:
    mm=ff_sort.loc[ff_sort['Country']==i]
    store_.append(list(mm.groupby('Items').sum()['Sales']))
store_

# `This looks fine but...`

we need to adjust for the fact that the Items are Unique and the Countries overlap. Since this is the case you have a few options:
+ Consider, using a conditional statement in a loop and plot (not straight forward)
+ Adjust each list to have same length, but you need to do elementwise comparisons
+ Plot each netsted barplot indiviudally (easy but, I wanna look cool. Don't you?)

**`I will choose to do option 02 above`**

*if you know an easier more efficient way, let me know!*

In [ ]:
from collections import defaultdict

fun=defaultdict(list)

for i in ff_sort.values:
    fun[i[0]].append(i[2])
fun
# ff_sort.values

In [ ]:
itm_updt=[]
itms=ff_sort['Items'].unique() # unique items

# create new DF with matching Items so we can create same length
# for X-axis later during plotting
# this will be appended to old DF
for i in fun.items():
    itm_updt.append([{'Items':set(itms)-set(i[1])},
{'Country':[i[0]]*len(set(itms)-set(i[1]))},
                {'Sales':[0]*len(set(itms)-set(i[1]))}])

# import json
vv=pd.DataFrame(itm_updt)

ll=pd.json_normalize(json.loads(vv.to_json(orient="records")))
ll.apply(pd.Series.explode).reset_index(drop=True)
# itm_updt
# vvll

+ This was needed in order to have each item correspond to each country for our plot.
    + Since, we are using nesting there needed to be aligned sales and country to each item.

In [ ]:
new_updated_info=ll.apply(pd.Series.explode).reset_index(drop=True)
new_updated_info.columns=['Items','Country','Sales']
new_updated_info.head()

In [ ]:
Hn=ff_sort.append(new_updated_info).\
sort_values(by=['Country','Sales'],
ascending=[True, False]).reset_index(drop=True)

Hn.head()
# list(Hn.groupby('Items').sum()['Sales'].index)

In [ ]:
st_=[]

ns_=Hn['Country'].unique()
for i in ns_:
    mm=Hn.loc[Hn['Country']==i]
    st_.append(list(mm.groupby('Items').sum()['Sales']))
st_

In [ ]:
order_x_labs=['ASSORTED COLOUR BIRD ORNAMENT',
 'HEART OF WICKER LARGE',
 'VINTAGE UNION JACK MEMOBOARD',
 'ManualManual',
 'RABBIT NIGHT LIGHT',
 'REGENCY TEAPOT ROSES',
 'ROUND SNACK BOXES SET OF 4 FRUITS',
 'SET OF TEA COFFEE SUGAR TINS PANTRY',
'METAL SIGN TAKE IT OR LEAVE IT']

In [ ]:
# This is the ordered data, manually because of difficulty with this plot!
pp=[4176.0,4121.64,3861.0,4161.06,3651.6,
    4992.0,4401.0,3794.4,7144.72,3828.0]

In [ ]:
# grid=[x for x in Hn.loc[:,'Items'].unique()]

grid=[x for x in order_x_labs]
graphs=st_
plt.figure(figsize=(8,8))
for gg,graph in enumerate(graphs):
        plt.bar(grid,graph,label=Hn.loc[:,'Country'].\
                unique()[gg],alpha=0.4)
                
plt.ylabel('Sales Total '+ '('+u'\u00A3'+')',size=25)
plt.title('Top 10 Items Sold & Country Which Sold Items',size=25)
plt.legend(loc="upper left")
plt.xticks(order_x_labs,rotation=45,ha="right",size=10)
plt.grid(True)

# putting labels above bars! crazyness...
for index, value in enumerate(pp[:5]):
    plt.text(index-.4,value, str(value))
for index, value in enumerate(pp[5:]):
    plt.text(index+3.6,value, str(value))

plt.show()

# `Pie Chart:`

In [ ]:
sales_tots=list(retail_sales_cleaned.\
        loc[:,['Country','Sales_Total']].\
     groupby(['Country']).sum()['Sales_Total'])

sales_round=[]
for i in sales_tots:
    sales_round.append(round(i,2))
sales_round[:5]
# retail_sales_cleaned['Country'].unique()[:10]
# retail_sales_cleaned.loc[:,['Country','Sales_Total']].\
#      groupby(['Country']).sum()[:5]
# sales_tots
# retail_sales_cleaned.\
#         loc[:,['Country','Sales_Total']].\
#      groupby(['Country']).sum()

In [ ]:
sales_tots=retail_sales_cleaned.loc[:,['Country','Sales_Total']].\
     groupby(['Country']).sum()
 
p=list(zip(sales_tots.index,sales_tots['Sales_Total']))
ii=pd.DataFrame(p,columns=['Country','Sales']).\
sort_values('Sales',ascending=False)[:10]
# p
ii

In [ ]:

oo=round(sum(ii['Sales'])/\
sum(retail_sales_cleaned['Sales_Total'])*100,3)
print('The Top 10 Countries make up only: %.2f percent of total sales'%oo)


In [ ]:
plt.figure(figsize=(11,11))

# Pie chart, where the slices will be ordered and plotted counter-clockwise:
labels = ii['Country']
sizes = ii['Sales']
explode = (0.1, 0,0, 0,0,0.9,1.8,2.3,2.9,3.9)

fig1, ax1 = plt.subplots(figsize=(11,11))
ax1.pie(sizes, labels=labels, autopct='%1.1f%%',explode=explode,
        shadow=True, startangle=40,radius=2,
        labeldistance = 1.03,pctdistance=.7)
ax1.axis('equal')  # Equal aspect ratio ensures that pie is drawn as a circle.
plt.title('Top 10 Countries By Sales Totals as Percent',size=20)
plt.show()

In [ ]:
# Pie chart, where the slices will be ordered and plotted counter-clockwise:
plt.figure(figsize=(9,9))
x = ii['Country']
y = ii['Sales']
colors = ['yellowgreen','red','gold','lightskyblue',
'white','lightcoral','blue','pink', 'darkgreen',
'yellow','grey','violet','magenta','cyan']
percent = 100.*y/y.sum()

patches, texts = plt.pie(y, colors=colors,
            startangle=90, radius=1.2)
labels = ['{0} - {1:1.2f} %'.format(i,j) for i,
          j in zip(x, percent)]

sort_legend = True
if sort_legend:
    patches, labels, dummy =  zip(*sorted(zip(patches, labels, y),
                                          key=lambda x: x[2],
                                          reverse=True))

plt.legend(patches, labels, bbox_to_anchor=(-0.1, 1.),
           fontsize=14)
plt.title('Top 10 Countries By Sales Totals as Percent',size=20)
plt.show()
# plt.savefig('piechart.png', bbox_inches='tight')

# <font color=red>Like</font>,Share &

# <font color=red>SUB</font>scribe

**`Help Support the Channel: Buy Me A Coffee @mrfugudatasci`**

# `Citations & Help:`

# ???

`Various Plotting Aids:`

https://jakevdp.github.io/PythonDataScienceHandbook/04.03-errorbars.html

https://realpython.com/pandas-plot-python/

http://aeturrell.com/2018/01/31/publication-quality-plots-in-python/

http://www.jesshamrick.com/2016/04/13/reproducible-plots/

https://www.stat.berkeley.edu/~nelle/teaching/2017-visualization/README.html

https://griddb.net/en/blog/data-visualization-with-python-matplotlib-and-griddb/

https://www.analyticsvidhya.com/blog/2020/02/beginner-guide-matplotlib-data-visualization-exploration-python/

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.plot.bar.html

https://www.pythoncharts.com/2019/03/26/grouped-bar-charts-matplotlib/

`Pie Plot Help:`

https://stackoverflow.com/questions/23577505/how-to-avoid-overlapping-of-labels-autopct-in-a-matplotlib-pie-chart

https://indianaiproduction.com/matplotlib-pie-chart/

https://matplotlib.org/3.3.3/gallery/pie_and_polar_charts/pie_and_donut_labels.html#sphx-glr-gallery-pie-and-polar-charts-pie-and-donut-labels-py

`Labels Above Bars in Bar Plot:`

https://pbpython.com/effective-matplotlib.html

https://stackoverflow.com/questions/62043788/add-values-above-bars-on-a-bar-chart-in-python

https://stackoverflow.com/questions/53066633/python-how-to-show-values-on-top-of-bar-plot

`Grouping & Multi-Level Dataframes:`

https://jakevdp.github.io/PythonDataScienceHandbook/03.08-aggregation-and-grouping.html

https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html